In [1]:
#Libreria

import pandas as pd #Para la manipulación de datos
from datetime import datetime  #Para la fecha de hoy
from selenium import webdriver #Para la navegación automática
import os #Para la manipulación de archivos
import time #Para la espera
import win32com.client as win32

#Importación de líbrerias de Selenium
from selenium.webdriver.common.by import By #Para la localización de elementos
from selenium.webdriver.support.ui import WebDriverWait #Para la espera de carga de elementos
from selenium.webdriver.support import expected_conditions as EC #Para la espera de carga de elementos
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException

In [2]:
#Extracción de datos de la plataforma SNIES
#Obtenemos el directorio actual
current_directory = os.getcwd()

#Función para clickear de forma segura
def safe_click(driver, locator):
    try:
        #Esperar a que cargue el elemento
        element = WebDriverWait(driver, 10).until(EC.element_to_be_clickable(locator))
        element.click()
    except (StaleElementReferenceException, TimeoutException):
        # Intentar nuevamente
        try:
            element = WebDriverWait(driver, 10).until(EC.element_to_be_clickable(locator))
            element.click()
        except (StaleElementReferenceException, TimeoutException):
            print(f"Failed to click element {locator}")

#Configuración del driver
options = webdriver.ChromeOptions() # Usamos chrome
prefs = {"download.default_directory" : os.path.join(current_directory, "Programas")}  # Descargar archivos dentro de esta carpeta

options.add_argument('--no-sandbox') # Seguridad
options.add_argument('--user-agent=""Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/74.0.3729.157 Safari/537.36""') # user agent
options.add_experimental_option("prefs",prefs)
driver = webdriver.Chrome(options=options)

#Navegamos a la página del SNIES
driver.get("https://hecaa.mineducacion.gov.co/consultaspublicas/programas")

driver.implicitly_wait(5) #Espera 5 segundos para cargar la página


#Identificadores de elementos a clickear
institucionFilter = (By.XPATH, '//*[@id="formFiltro:j_idt33"]/tbody/tr[2]/td/div/div[2]') #Filtro de estado de institución (Activo)
programaFilter = (By.XPATH, '//*[@id="formFiltro:j_idt65"]/tbody/tr[2]/td/div/div[2]') #Filtro de estado de programa (Activo)
academicoFilter = (By.XPATH, '//*[@id="formFiltro:j_idt102"]/tbody/tr[3]/td/div/div[2]') #Filtro de tipo académico (Posgrado)
formacionFilter = (By.XPATH, '//*[@id="formFiltro:j_idt108"]/tbody/tr[1]/td/div/div[2]') #Filtro de tipo de formación (Todos)
downloadButton = (By.XPATH, '//*[@id="j_idt153:j_idt155"]') #Botón de descarga

#Clickeamos los filtros
safe_click(driver, institucionFilter)
time.sleep(3) 
safe_click(driver, programaFilter)
time.sleep(3)
safe_click(driver, academicoFilter)
time.sleep(3)
safe_click(driver, formacionFilter)
time.sleep(3)
safe_click(driver, downloadButton)

#Espera a que se descargue el archivo para ejecutar el resto del código
while (not os.path.exists("Programas\Programas.xlsx")):
    print("Descargando archivo...")
    time.sleep(5)
driver.quit()

<>:53: SyntaxWarning: invalid escape sequence '\P'
<>:53: SyntaxWarning: invalid escape sequence '\P'
C:\Users\hasuarez\AppData\Local\Temp\ipykernel_4692\4021299441.py:53: SyntaxWarning: invalid escape sequence '\P'
  while (not os.path.exists("Programas\Programas.xlsx")):
C:\Users\hasuarez\AppData\Local\Temp\ipykernel_4692\4021299441.py:53: SyntaxWarning: invalid escape sequence '\P'
  while (not os.path.exists("Programas\Programas.xlsx")):


ProtocolError: ('Connection aborted.', ConnectionResetError(10054, 'Se ha forzado la interrupción de una conexión existente por el host remoto', None, 10054, None))

In [4]:
#Fecha de hoy
today = datetime.today().strftime('%d-%m-%y')
#Fecha del último archivo
last = "10-11-25"
programaNombre = os.path.join(current_directory, "Programas", "Programas.xlsx")
programaNuevoNombre = os.path.join(current_directory, "Programas", "Programas postgrado " + today + ".xlsx")

#Renombramos el archivo descargado con la fecha de hoy (Si Existe)
try:
    os.rename(programaNombre, programaNuevoNombre)
except:
    print("No existe archivo actual")

#Leemos el archivo
df = pd.read_excel("Programas\Programas postgrado "+ today +".xlsx", sheet_name="Programas")

#Leemos el archivo más (Si Existe)
try:
    dfpast = pd.read_excel("Programas\Programas postgrado "+ last +".xlsx", sheet_name="Programas")
    #dfpast=pd.read_excel('Programas 23-02-24.xlsx', sheet_name="Programas")
except:
    print("No existe archivo antiguo para comparar")

#Leemos el archivo de categorización de divisiones Uninorte
categories = pd.read_excel("Categorización divisiones SNIES .xlsx", sheet_name="Hoja3")

#Columnas a usar para el análisis
finalColumns = [
    'CÓDIGO_INSTITUCIÓN',
    'NOMBRE_INSTITUCIÓN',
    'SECTOR',
    'DEPARTAMENTO_OFERTA_PROGRAMA',
    'MUNICIPIO_OFERTA_PROGRAMA',
    'CÓDIGO_SNIES_DEL_PROGRAMA',
    'NOMBRE_DEL_PROGRAMA',
    'MODALIDAD',
    'NÚMERO_CRÉDITOS',
    'NÚMERO_PERIODOS_DE_DURACIÓN',
    'PERIODICIDAD',
    'COSTO_MATRÍCULA_ESTUD_NUEVOS',
    'PERIODICIDAD_ADMISIONES',
    'FECHA_DE_REGISTRO_EN_SNIES',
    'CINE_F_2013_AC_CAMPO_AMPLIO', 
    'CINE_F_2013_AC_CAMPO_ESPECÍFIC',
    'CINE_F_2013_AC_CAMPO_DETALLADO',
    'NÚCLEO_BÁSICO_DEL_CONOCIMIENTO',
    'NIVEL_DE_FORMACIÓN']


droppedDf = df[finalColumns] #Filtramos las columnas del archivo más reciente
droppedDfpast = dfpast[finalColumns] #Filtramos las columnas del archivo más antiguo

#Quitamos el aviso del SNIES
droppedDf = droppedDf.iloc[:-2]
droppedDfpast = droppedDfpast.iloc[:-2]


<>:15: SyntaxWarning: invalid escape sequence '\P'
<>:19: SyntaxWarning: invalid escape sequence '\P'
<>:15: SyntaxWarning: invalid escape sequence '\P'
<>:19: SyntaxWarning: invalid escape sequence '\P'
C:\Users\ecpereira\AppData\Local\Temp\ipykernel_81240\1376400699.py:15: SyntaxWarning: invalid escape sequence '\P'
  df = pd.read_excel("Programas\Programas postgrado "+ today +".xlsx", sheet_name="Programas")
C:\Users\ecpereira\AppData\Local\Temp\ipykernel_81240\1376400699.py:19: SyntaxWarning: invalid escape sequence '\P'
  dfpast = pd.read_excel("Programas\Programas postgrado "+ last +".xlsx", sheet_name="Programas")
c:\Users\ecpereira\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\ecpereira\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\styles\stylesheet.py:237: Us

In [5]:
#Eliminamos los programas sin código de programa y convertimos a entero
droppedDf['CÓDIGO_SNIES_DEL_PROGRAMA'] = droppedDf['CÓDIGO_SNIES_DEL_PROGRAMA'].dropna().astype(int)
droppedDfpast['CÓDIGO_SNIES_DEL_PROGRAMA'] = droppedDfpast['CÓDIGO_SNIES_DEL_PROGRAMA'].dropna().astype(int)

#Rellenamos los valores nulos con 0 y convertimos a entero
droppedDf['NÚMERO_CRÉDITOS'] = droppedDf['NÚMERO_CRÉDITOS'].fillna(0).astype(int)
droppedDfpast['NÚMERO_CRÉDITOS'] = droppedDfpast['NÚMERO_CRÉDITOS'].fillna(0).astype(int)

#Creamos un código para identificar el programa en caso que cambie el número de créditos
droppedDf["PROGINSTI"] = droppedDf["CÓDIGO_SNIES_DEL_PROGRAMA"].astype(str)+"-"+droppedDf["NÚMERO_CRÉDITOS"].astype(str)
droppedDfpast["PROGINSTI"] =droppedDfpast["CÓDIGO_SNIES_DEL_PROGRAMA"].astype(str)+"-"+droppedDfpast["NÚMERO_CRÉDITOS"].astype(str)

#Fecha de obtención
newDate = pd.to_datetime(today, format='%d-%m-%y')
newDateFormatted = newDate.strftime('%d/%m/%Y')


In [6]:
#Los programas inactivos son aquellos que existen en el archivo antiguo pero no en el nuevo
eliminadosDF = droppedDfpast[~droppedDfpast["PROGINSTI"].isin(droppedDf["PROGINSTI"])]
#Los programas nuevos son aquellos que existen en el archivo nuevo pero no en el antiguo
nuevosDF = droppedDf[~droppedDf["PROGINSTI"].isin(droppedDfpast["PROGINSTI"])]
#Los programas modificados son aquellos que existen en ambos documentos ya que existen tanto en el antiguo como en el nuevo pero con cambios en los créditos
modificadosDF = eliminadosDF[eliminadosDF['CÓDIGO_SNIES_DEL_PROGRAMA'].isin(nuevosDF['CÓDIGO_SNIES_DEL_PROGRAMA'])]

#Funciones que tiene un posible error
#modificadosDF["CREDITOS_NUEVOS"] = droppedDf['NÚMERO_CRÉDITOS']
#modificadosDF["CREDITOS_ANTERIORES"] = droppedDfpast['NÚMERO_CRÉDITOS']

#Posible solucion a el error
modificadosDF = modificadosDF.merge(
    droppedDf[['CÓDIGO_SNIES_DEL_PROGRAMA', 'NÚMERO_CRÉDITOS']],
    on='CÓDIGO_SNIES_DEL_PROGRAMA',
    how='left',
    suffixes=('', '_NUEVOS')
)
modificadosDF = modificadosDF.merge(
    droppedDfpast[['CÓDIGO_SNIES_DEL_PROGRAMA', 'NÚMERO_CRÉDITOS']],
    on='CÓDIGO_SNIES_DEL_PROGRAMA',
    how='left',
    suffixes=('', '_ANTERIORES')
)

modificadosDF.dropna(subset='NÚMERO_CRÉDITOS')

#Eliminamos los programas que se encuentran en los modificados para que no se repitan
eliminadosDF = eliminadosDF[~eliminadosDF["CÓDIGO_SNIES_DEL_PROGRAMA"].isin(modificadosDF["CÓDIGO_SNIES_DEL_PROGRAMA"])]
nuevosDF = nuevosDF[~nuevosDF["CÓDIGO_SNIES_DEL_PROGRAMA"].isin(modificadosDF["CÓDIGO_SNIES_DEL_PROGRAMA"])]

#Agregamos la fecha de obtención y convertimos a fecha la fecha de registro en SNIES
eliminadosDF['FECHA_OBTENCION'] = newDateFormatted
nuevosDF['FECHA_OBTENCION'] = newDateFormatted
modificadosDF['FECHA_OBTENCION'] = newDateFormatted
eliminadosDF["FECHA_DE_REGISTRO_EN_SNIES"] = eliminadosDF["FECHA_DE_REGISTRO_EN_SNIES"].dt.date
nuevosDF["FECHA_DE_REGISTRO_EN_SNIES"] = nuevosDF["FECHA_DE_REGISTRO_EN_SNIES"].dt.date
modificadosDF["FECHA_DE_REGISTRO_EN_SNIES"] = modificadosDF["FECHA_DE_REGISTRO_EN_SNIES"].dt.date


In [7]:
#Chequeo de longitudes
print(len(droppedDf))
print(len(droppedDfpast))
print(len(eliminadosDF))
print(len(nuevosDF))
print(len(modificadosDF))

8508
8508
0
0
0


# Agregación de Divisiones Uninorte

In [8]:
#Agregamos la división Uninorte a los programas a partir del CINE de Campo detallado en los 3 archivos
nuevosDFMerge = nuevosDF.merge(on=["CINE_F_2013_AC_CAMPO_DETALLADO"], right= categories[["CINE_F_2013_AC_CAMPO_DETALLADO","DIVISIÓN UNINORTE"]], how="left")
eliminadosDFMerge = eliminadosDF.merge(on=["CINE_F_2013_AC_CAMPO_DETALLADO"], right= categories[["CINE_F_2013_AC_CAMPO_DETALLADO","DIVISIÓN UNINORTE"]], how="left")
modificadosDFMerge = modificadosDF.merge(on=["CINE_F_2013_AC_CAMPO_DETALLADO"], right= categories[["CINE_F_2013_AC_CAMPO_DETALLADO","DIVISIÓN UNINORTE"]], how="left")

#modificadosDFMerge
nuevosDFMerge 
#eliminadosDFMerge

,CÓDIGO_INSTITUCIÓN,NOMBRE_INSTITUCIÓN,SECTOR,DEPARTAMENTO_OFERTA_PROGRAMA,MUNICIPIO_OFERTA_PROGRAMA,CÓDIGO_SNIES_DEL_PROGRAMA,NOMBRE_DEL_PROGRAMA,MODALIDAD,NÚMERO_CRÉDITOS,NÚMERO_PERIODOS_DE_DURACIÓN,...,PERIODICIDAD_ADMISIONES,FECHA_DE_REGISTRO_EN_SNIES,CINE_F_2013_AC_CAMPO_AMPLIO,CINE_F_2013_AC_CAMPO_ESPECÍFIC,CINE_F_2013_AC_CAMPO_DETALLADO,NÚCLEO_BÁSICO_DEL_CONOCIMIENTO,NIVEL_DE_FORMACIÓN,PROGINSTI,FECHA_OBTENCION,DIVISIÓN UNINORTE


# Concatenación/Creación de Archivos Finales

In [9]:
if os.path.exists(r"C:\Users\ecpereira\Universidad del Norte\ESTUDIOS UNINORTE - Documentos\General\Novedades SNIES\Inactivos posgrado.xlsx"):
    #Si el archivo existe, se le concatenan los programas eliminados y se eliminan los repetidos
    existing_df = pd.read_excel(r"C:\Users\ecpereira\Universidad del Norte\ESTUDIOS UNINORTE - Documentos\General\Novedades SNIES\Inactivos posgrado.xlsx", sheet_name='Sheet1')
    combined_df = pd.concat([existing_df, eliminadosDFMerge])
    final_df = combined_df.drop_duplicates(subset=['PROGINSTI'])
    final_df.to_excel(r"C:\Users\ecpereira\Universidad del Norte\ESTUDIOS UNINORTE - Documentos\General\Novedades SNIES\Inactivos posgrado.xlsx",index=False)
else:
    #Si el archivo no existe, se crea
    eliminadosDFMerge.to_excel(r"C:\Users\ecpereira\Universidad del Norte\ESTUDIOS UNINORTE - Documentos\General\Novedades SNIES\Inactivos posgrado.xlsx", index=False)

if os.path.exists(r"C:\Users\ecpereira\Universidad del Norte\ESTUDIOS UNINORTE - Documentos\General\Novedades SNIES\Nuevos posgrado.xlsx"):
    #Si el archivo existe, se le concatenan los programas nuevos y se eliminan los repetidos
    existing_df = pd.read_excel(r"C:\Users\ecpereira\Universidad del Norte\ESTUDIOS UNINORTE - Documentos\General\Novedades SNIES\Nuevos posgrado.xlsx", sheet_name='Sheet1')
    combined_df = pd.concat([existing_df, nuevosDFMerge])
    final_df = combined_df.drop_duplicates(subset=['PROGINSTI'])
    final_df.to_excel(r"C:\Users\ecpereira\Universidad del Norte\ESTUDIOS UNINORTE - Documentos\General\Novedades SNIES\Nuevos posgrado.xlsx", index=False)
else:
    #Si el archivo no existe, se crea
    nuevosDFMerge.to_excel(r"C:\Users\ecpereira\Universidad del Norte\ESTUDIOS UNINORTE - Documentos\General\Novedades SNIES\Nuevos posgrado.xlsx", index=False)

if os.path.exists(r"C:\Users\ecpereira\Universidad del Norte\ESTUDIOS UNINORTE - Documentos\General\Novedades SNIES\Modificados posgrado.xlsx"):
    #Si el archivo existe, se le concatenan los programas modificados y se eliminan los repetidos
    existing_df = pd.read_excel(r"C:\Users\ecpereira\Universidad del Norte\ESTUDIOS UNINORTE - Documentos\General\Novedades SNIES\Modificados posgrado.xlsx", sheet_name='Sheet1')
    combined_df = pd.concat([existing_df, modificadosDFMerge])
    final_df = combined_df.drop_duplicates(subset=['PROGINSTI'])
    final_df.to_excel(r"C:\Users\ecpereira\Universidad del Norte\ESTUDIOS UNINORTE - Documentos\General\Novedades SNIES\Modificados posgrado.xlsx", index=False)
else:
    #Si el archivo no existe, se crea
    modificadosDFMerge.to_excel(r"C:\Users\ecpereira\Universidad del Norte\ESTUDIOS UNINORTE - Documentos\General\Novedades SNIES\Modificados posgrado.xlsx", index=False)


C:\Users\ecpereira\AppData\Local\Temp\ipykernel_81240\2905785862.py:4: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = pd.concat([existing_df, eliminadosDFMerge])
C:\Users\ecpereira\AppData\Local\Temp\ipykernel_81240\2905785862.py:14: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = pd.concat([existing_df, nuevosDFMerge])
C:\Users\ecpereira\AppData\Local\Temp\ipykernel_81240\2905785862.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entri

In [10]:
data_intantivo = pd.read_excel(r"C:\Users\ecpereira\Universidad del Norte\ESTUDIOS UNINORTE - Documentos\General\Novedades SNIES\Inactivos posgrado.xlsx", sheet_name='Sheet1')
data_nuevos = pd.read_excel(r"C:\Users\ecpereira\Universidad del Norte\ESTUDIOS UNINORTE - Documentos\General\Novedades SNIES\Nuevos posgrado.xlsx", sheet_name='Sheet1')  
data_modificados = pd.read_excel(r"C:\Users\ecpereira\Universidad del Norte\ESTUDIOS UNINORTE - Documentos\General\Novedades SNIES\Modificados posgrado.xlsx", sheet_name='Sheet1')

#Agregamos una columna de estado a los DataFrames
data_intantivo["Estado"]=["Activo" if x in droppedDf["CÓDIGO_SNIES_DEL_PROGRAMA"].values else "Inactivo" for x in data_intantivo["CÓDIGO_SNIES_DEL_PROGRAMA"]]
data_nuevos["Estado"]=["Activo" if x in droppedDf["CÓDIGO_SNIES_DEL_PROGRAMA"].values else "Inactivo" for x in data_nuevos["CÓDIGO_SNIES_DEL_PROGRAMA"]]
data_modificados["Estado"]=["Activo" if x in droppedDf["CÓDIGO_SNIES_DEL_PROGRAMA"].values else "Inactivo" for x in data_modificados["CÓDIGO_SNIES_DEL_PROGRAMA"]]

#Exportamos los archivos a Excel
data_intantivo.to_excel(r"C:\Users\ecpereira\Universidad del Norte\ESTUDIOS UNINORTE - Documentos\General\Novedades SNIES\Inactivos posgrado.xlsx",index=False)
data_nuevos.to_excel(r"C:\Users\ecpereira\Universidad del Norte\ESTUDIOS UNINORTE - Documentos\General\Novedades SNIES\Nuevos posgrado.xlsx",index=False)
data_modificados.to_excel(r"C:\Users\ecpereira\Universidad del Norte\ESTUDIOS UNINORTE - Documentos\General\Novedades SNIES\Modificados posgrado.xlsx",index=False)

In [11]:
#Cliente de Outlook
outlook = win32.Dispatch('outlook.application')

#Ruta de los archivos inactivos, nuevos y modificados
eliminados  = r'C:\Users\ecpereira\Universidad del Norte\ESTUDIOS UNINORTE - Documentos\General\Novedades SNIES\Inactivos posgrado.xlsx'
nuevos = r'C:\Users\ecpereira\Universidad del Norte\ESTUDIOS UNINORTE - Documentos\General\Novedades SNIES\Nuevos posgrado.xlsx'
modificados = r'C:\Users\ecpereira\Universidad del Norte\ESTUDIOS UNINORTE - Documentos\General\Novedades SNIES\Modificados posgrado.xlsx'

#Correo único para todos los destinatarios
mail = outlook.CreateItem(0) #Creación del correo
mail.Subject = 'Nuevos/Inactivos/Modificados '+today #Asunto del correo
mail.HTMLBody = '<h2>Reporte mensual Programas Nuevos/Inactivos/Modificados posgrado</h2>' #Cuerpo del correo
mail.Attachments.Add(eliminados) #Agregamos los archivos al correo
mail.Attachments.Add(nuevos)
mail.Attachments.Add(modificados)
mail.To = 'mbastidas@uninorte.edu.co; degutierrez@uninorte.edu.co; lmlayes@uninorte.edu.co' #Todos los destinatarios
mail.Send() #Envío del correo                                                       